In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import sqlite3

Loading the dataset 

In [3]:
df=pd.read_csv("train.csv")

In [4]:


df = pd.read_csv('train.csv')


customers = df[['Customer ID', 'Customer Name', 'Segment', 'Country']] \
    .drop_duplicates(subset='Customer ID') \
    .reset_index(drop=True)


orders = df[['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID',
             'City', 'State', 'Postal Code', 'Region']] \
    .drop_duplicates(subset='Order ID') \
    .reset_index(drop=True)


products = df[['Product ID', 'Category', 'Sub-Category', 'Product Name']] \
    .drop_duplicates(subset='Product ID') \
    .reset_index(drop=True)

order_details = df[['Row ID', 'Order ID', 'Product ID', 'Sales']] \
    .reset_index(drop=True)

rebuilt = order_details.merge(orders, on='Order ID') \
                        .merge(customers, on='Customer ID') \
                        .merge(products, on='Product ID')

assert rebuilt.shape[0] == df.shape[0], "Row count mismatch!"
assert round(rebuilt['Sales'].sum(), 2) == round(df['Sales'].sum(), 2), "Sales total mismatch!"
print("✅ Data verified — no loss after normalization")


conn = sqlite3.connect('superstore_sales_fixed.db')
customers.to_sql('customers', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
order_details.to_sql('order_details', conn, if_exists='replace', index=False)
conn.commit()


print("✅ Database saved: superstore_sales_fixed.db")

✅ Data verified — no loss after normalization
✅ Database saved: superstore_sales_fixed.db


In [5]:

customers = df[['Customer ID', 'Customer Name', 'Segment', 'Country']] \
    .drop_duplicates(subset='Customer ID') \
    .reset_index(drop=True)

orders = df[['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID',
             'City', 'State', 'Postal Code', 'Region']] \
    .drop_duplicates(subset='Order ID') \
    .reset_index(drop=True)

products = df[['Product ID', 'Category', 'Sub-Category', 'Product Name']] \
    .drop_duplicates(subset='Product ID') \
    .reset_index(drop=True)

order_details = df[['Row ID', 'Order ID', 'Product ID', 'Sales']] \
    .reset_index(drop=True)


rebuilt = order_details.merge(orders, on='Order ID') \
                        .merge(customers, on='Customer ID') \
                        .merge(products, on='Product ID')

assert rebuilt.shape[0] == df.shape[0], "Row count mismatch!"
assert round(rebuilt['Sales'].sum(), 2) == round(df['Sales'].sum(), 2), "Sales total mismatch!"
print("✅ Data verified — no loss after split")

customers.to_csv('customers.csv', index=False)
orders.to_csv('orders.csv', index=False)
products.to_csv('products.csv', index=False)
order_details.to_csv('order_details.csv', index=False)

print("✅ 4 CSV files created:")
print("customers:", customers.shape)
print("orders:", orders.shape)
print("products:", products.shape)
print("order_details:", order_details.shape)

✅ Data verified — no loss after split
✅ 4 CSV files created:
customers: (793, 4)
orders: (4922, 9)
products: (1861, 4)
order_details: (9800, 4)


Data cleaning 

There are a datatpye error in df ,the columns order is and ship date has to in date datatypt ,so now i  will change the data type 

In [6]:
df[['Order Date','Ship Date']]=df[['Order Date','Ship Date']].apply(pd.to_datetime,format='mixed')
df.describe()

,Row ID,Order Date,Ship Date,Postal Code,Sales
count,9800.000000,9800,9800,9789.000000,9800.000000
mean,4900.500000,2017-04-12 14:24:35.265306,2017-04-21 19:45:12.489796,55273.322403,230.769059
min,1.000000,2015-01-02 00:00:00,2015-01-04 00:00:00,1040.000000,0.444000
25%,2450.750000,2016-05-02 12:00:00,2016-05-08 00:00:00,23223.000000,17.248000
50%,4900.500000,2017-05-30 00:00:00,2017-06-12 00:00:00,58103.000000,54.490000
75%,7350.250000,2018-04-11 00:00:00,2018-05-02 00:00:00,90008.000000,210.605000
max,9800.000000,2018-12-30 00:00:00,2019-05-01 00:00:00,99301.000000,22638.480000
std,2829.160653,NaN,NaN,32041.223413,626.651875


In [7]:
df.isnull().sum()
df.duplicated().sum()
# There is no duplicate value in df 


np.int64(0)

In [8]:
df.describe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9800 non-null   int64         
 1   Order ID       9800 non-null   str           
 2   Order Date     9800 non-null   datetime64[us]
 3   Ship Date      9800 non-null   datetime64[us]
 4   Ship Mode      9800 non-null   str           
 5   Customer ID    9800 non-null   str           
 6   Customer Name  9800 non-null   str           
 7   Segment        9800 non-null   str           
 8   Country        9800 non-null   str           
 9   City           9800 non-null   str           
 10  State          9800 non-null   str           
 11  Postal Code    9789 non-null   float64       
 12  Region         9800 non-null   str           
 13  Product ID     9800 non-null   str           
 14  Category       9800 non-null   str           
 15  Sub-Category   9800 non-null   s

connecting sql 

In [9]:
#total orders 
q1="""
select count("order id") as total_orders  from orders
"""
pd.read_sql(q1,conn)

,total_orders
0,4922


 total_sales 

In [10]:
# Total sales 
q1=""" 
select sum(sales) as total_sales from order_details

"""
total_sales=pd.read_sql(q1,conn)

pd.set_option('display.float_format', '{:,.2f}'.format)
total_sales

,total_sales
0,"2,261,536.78"


In [11]:
q1 = """
select substr(o."Order Date", -4) as year,
       sum(od.Sales) as total_sales
from orders as o 
join order_details as od using("Order ID")
group by year
order by year
"""
pd.read_sql(q1, conn)

,year,total_sales
0,2015,"479,856.21"
1,2016,"459,436.01"
2,2017,"600,192.55"
3,2018,"722,052.02"


### Key Insights

- Sales declined slightly from 479,856.21 in 2015 to 459,436.01 in 2016, representing a decrease of approximately 4.3%.
- Sales increased significantly in 2017, reaching 600,192.55, which was approximately 30.6% higher than the previous year.
- The strongest growth occurred in 2018, when sales reached 722,052.02, an increase of approximately 20.3% compared to 2017.
- 2018 recorded the highest sales during the analyzed period, while 2016 had the lowest sales.
- Overall, sales increased by approximately 50.5% between 2015 and 2018, indicating strong long-term business growth despite the temporary decline in 2016.
- The consistent growth observed from 2016 to 2018 suggests that the business gained significant momentum during the later years of the analysis.

### Business Recommendation

The company should investigate the factors behind the strong sales growth from 2016 to 2018 and identify the products, customer segments, and regions that contributed most to this growth. These successful strategies can be further leveraged to sustain future business performance.

In [12]:
q1 = """
SELECT COUNT(DISTINCT o."Order ID") as total_orders,
       COUNT(DISTINCT o."Customer ID") as total_customers,
       ROUND(SUM(od.sales),2) as total_sales
FROM order_details od
JOIN orders o ON od."Order ID" = o."Order ID"
"""
print(pd.read_sql(q1, conn))

   total_orders  total_customers  total_sales
0          4922              793 2,261,536.78


.........................................product wise sales................................................... 

In [13]:
q="""
select 
p.category,sum(od.sales) as total_sales
from products as p join order_details as od using("product Id")
group by p.category
order by total_sales
"""
pd.read_sql(q,conn)

,Category,total_sales
0,Office Supplies,"705,422.33"
1,Furniture,"728,658.58"
2,Technology,"827,455.87"


Category Technology has highest sales 

.......................................sub category wise sales....................................................

In [14]:
q="""
select 
p.category,
p.'sub-category',sum(od.sales) as total_sales
from products as p join order_details as od using("product Id")
group by p.'sub-category'
order by total_sales desc
"""
sub_category_wise_sales=pd.read_sql(q,conn)
sub_category_wise_sales

,Category,Sub-Category,total_sales
0,Technology,Phones,"327,782.45"
1,Furniture,Chairs,"322,822.73"
2,Office Supplies,Storage,"219,343.39"
3,Furniture,Tables,"202,810.63"
4,Office Supplies,Binders,"200,028.79"
5,Technology,Machines,"189,238.63"
6,Technology,Accessories,"164,186.70"
7,Technology,Copiers,"146,248.09"
8,Furniture,Bookcases,"113,813.20"
9,Office Supplies,Appliances,"104,618.40"


Key Business Insights
Technology is a strong-performing category, with Phones generating the highest sales at 327,782.45.
Within the Furniture category, Chairs are the top-performing sub-category, generating 322,822.73 in sales.
In Office Supplies, Storage leads with 219,343.39 in sales, followed by Binders with 200,028.79.
The top-performing sub-categories overall are Phones (327,782.45) and Chairs (322,822.73), indicating strong customer demand for these products.
Fasteners is the lowest-performing sub-category, with only 3,001.96 in sales, followed by Labels (12,347.73) and Envelopes (16,128.05).
Phones and Chairs generate almost similar sales, with Phones slightly outperforming Chairs by approximately 4,960.
The business should focus on high-performing sub-categories such as Phones, Chairs, Storage, and Binders, while reviewing the demand, pricing, and inventory strategy for low-performing sub-categories.

...........................................................Top customers..................................................................

In [15]:
q = """
select
c."customer name", c.segment, sum(od.sales) as total_sales
from customers c 
join orders o using("customer id") 
join order_details od using("order id")
group by c."customer id",
    c."customer name",
    c.segment
order by total_sales desc limit 10
"""
top_customers = pd.read_sql(q, conn)
top_customers

,Customer Name,Segment,total_sales
0,Sean Miller,Home Office,"25,043.05"
1,Tamara Chand,Corporate,"19,052.22"
2,Raymond Buch,Consumer,"15,117.34"
3,Tom Ashbrook,Home Office,"14,595.62"
4,Adrian Barton,Consumer,"14,473.57"
5,Ken Lonsdale,Consumer,"14,175.23"
6,Sanjit Chand,Consumer,"14,142.33"
7,Hunter Lopez,Consumer,"12,873.30"
8,Sanjit Engle,Consumer,"12,209.44"
9,Christopher Conant,Consumer,"12,129.07"


- Sean Miller is the highest-value customer, generating total sales of 25,043.05, significantly ahead of the other top customers.
- Tamara Chand ranks second with sales of 19,052.22, followed by Raymond Buch with 15,117.34.
- The top two customers, Sean Miller and Tamara Chand, generated substantially higher sales than most other customers in the top 10.
- The majority of the top 10 customers belong to the Consumer segment, indicating that individual consumers represent a significant share of high-value customers.
- The Consumer segment accounts for 7 of the top 10 customers, while the remaining customers belong to the Corporate and Home Office segments.
- Sean Miller's sales are approximately 31% higher than those of the second-highest customer, Tamara Chand, highlighting a particularly valuable customer relationship.
- The sales difference between the highest-performing customer (Sean Miller) and the tenth-ranked customer (Christopher Conant) is approximately 12,913.98.
- The presence of multiple high-value customers across different segments suggests opportunities for targeted customer retention and segment-specific marketing strategies.

................................ Best selling states......................................................

In [16]:
q="""
select
o.state,sum(od.sales) as total_sales
from customers c  join orders o using("customer id") join order_details od using("order id")
group by o.state 
order by total_sales desc limit 10

"""
top_selling_states=pd.read_sql(q,conn)
top_selling_states

,State,total_sales
0,California,"446,306.46"
1,New York,"306,361.15"
2,Texas,"168,572.53"
3,Washington,"135,206.85"
4,Pennsylvania,"116,276.65"
5,Florida,"88,436.53"
6,Illinois,"79,236.52"
7,Michigan,"76,136.07"
8,Ohio,"75,130.35"
9,Virginia,"70,636.72"


    State	total_sales
0	California	446,306.46
1	New York	306,361.15
2	Texas	168,572.53
3	Washington	135,206.85
4	Pennsylvania	116,276.65
5	Florida	88,436.53
6	Illinois	79,236.52
7	Michigan	76,136.07
8	Ohio	75,130.35
9	Virginia	70,636.721.

Top 2 states drive ~33% of total sales
California ($446K) + New York ($306K) = ~$752K out of the company's total sales of ~$2.26M. The business is heavily concentrated in just these two states.

2. California outsells New York by 46%, despite fewer differences in market size
This gap raises the question — is California's marketing/reach stronger, or is New York still an underdeveloped market with growth potential?

3. Average order value tells a different story than total sales
- California: 1,002 orders → $446,306 sales → ~$445/order
- New York: 547 orders → $306,361 sales → ~$560/order
- Texas: 480 orders → $168,572 sales → ~$351/order

New York has fewer orders but a higher average order value — customers there tend to buy bigger or pricier orders. This suggests New York could be a good market to push premium products.

4. Mid-tier states show room for growth
Ohio, Michigan, and Virginia have similar sales totals (~$70–76K). Targeted marketing could help push them up to the level of Florida or Illinois.

5. Long tail — most states contribute very little
These top 10 states account for the bulk of total sales; the remaining ~39 states/territories contribute much less. Worth investigating whether this is due to low demand or weak distribution/marketing in those regions.

Less selling states

In [17]:
q="""
select
o.state,sum(od.sales) as total_sales
from customers c  join orders o using("customer id") join order_details od using("order id")
group by o.state 
order by total_sales  limit 5

"""
top_selling_states=pd.read_sql(q,conn)
top_selling_states

,State,total_sales
0,North Dakota,919.91
1,West Virginia,"1,209.82"
2,Maine,"1,270.53"
3,South Dakota,"1,315.56"
4,Wyoming,"1,603.14"


Insight:

North Dakota is the lowest (~$920) — ~485x less than California ($446,306)
The combined total of all 5 of these states is barely more than a single average customer's lifetime spend ($500–800 range)
These are likely smaller/rural states with lower population, or areas where the company's presence/distribution is weak

Two possible business reasons:

Low demand — the market itself is small (fewer people/businesses to sell to)
Weak reach — the company's marketing or distribution network hasn't properly reached these states

.................................Top Selling States..........................................................

In [18]:
q="""
select
    o.state,
    count(distinct o."order id") as total_order
from customers c 
join orders o using("customer id") 
group by o.state 
order by total_order desc 
limit 10

"""
top_selling_states=pd.read_sql(q,conn)
top_selling_states

,State,total_order
0,California,1002
1,New York,547
2,Texas,480
3,Pennsylvania,285
4,Illinois,270
5,Washington,254
6,Ohio,227
7,Florida,197
8,North Carolina,134
9,Michigan,116


✅ Total Sales
✅ Total Profit
✅ Total Orders
✅ Total Customers
✅ Average Order Value
✅ Year-wise Sales
✅ Year-wise Profit
✅ Category-wise Sales
✅ Category-wise Profit
✅ Sub-category-wise Sales
✅ Sub-category-wise Profit
✅ Top 10 Customers
✅ Top 10 Products
✅ Region-wise Sales
✅ State-wise Profit
✅ Ship Mode Analysis
✅ Loss-making Products
✅ Loss-making States

..............................................Monthly Sales trend...................................................... 

In [26]:
q = """
select 
    substr(o."Order Date", 4, 2) || '-' || substr(o."Order Date", -4) as month_year,
    sum(od.Sales) as total_sales
from orders as o 
join order_details as od using("Order ID")
group by substr(o."Order Date", -4), substr(o."Order Date", 4, 2)
order by substr(o."Order Date", -4), substr(o."Order Date", 4, 2)
"""
monthly_sales = pd.read_sql(q, conn)
#monthly_sales

..........................................ship mode...........................................................

In [ ]:
q=""" 
select o."ship mode",
count(distinct o."order id") as order_count,
sum(od.sales)
from orders o join order_details od using('order id')
group by o."ship mode"
order by order_count desc
"""
prefferred_shiping_mode=pd.read_sql(q,conn)

,Ship Mode,order_count,sum(od.sales)
0,Standard Class,2945,"1,340,831.31"
1,Second Class,944,"449,914.18"
2,First Class,772,"345,572.26"
3,Same Day,261,"125,219.04"


Insights:
Standard Class is the most preferred shipping mode, with 2,945 orders and $1.34M in sales.
Second Class ranks second, generating $449.91K in sales from 944 orders.
First Class generated $345.57K in sales with 772 orders.
Same Day is the least-used shipping mode, with only 261 orders and $125.22K in sales.
Overall Insight: Customers mostly prefer Standard Class shipping, indicating that cost-effective delivery options are more popular than faster delivery options.